13 - Human in the Loop (Approvals)

This notebook converts the Python tutorial `$13 human in the loop` into an interactive Jupyter format.

This section sets up the imports and environment configuration needed for the tutorial.

In [ ]:
"""13 - Human in the Loop (Approvals)==================================- `interrupt_on` pauses the agent BEFORE a risky tool runs- The run stops and returns `result["__interrupt__"]` describing what it wants- You resume with `Command(resume={"decisions": [...]})`- Requires a checkpointer — the paused run has to be stored somewhereRun:  python 13-human_in_the_loop.py"""

This section implements imports and setup.

In [ ]:
# %% Step 1: Imports and setupimport osfrom deepagents import create_deep_agentfrom dotenv import load_dotenvfrom langgraph.checkpoint.memory import InMemorySaverfrom langgraph.types import Command  # carries the human's decision back infrom rich import printload_dotenv()MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")def text_of(message) -> str:    """Normalize message content across providers (str vs content blocks)."""    content = message.content    if isinstance(content, str):        return content    return "".join(block.get("text", "") for block in content if isinstance(block, dict))

This section implements a tool with real-world consequences.

In [ ]:
# %% Step 2: A tool with real-world consequencesdef send_email(to: str, body: str) -> str:    """Send an email to a recipient.    Args:        to: The recipient's email address.        body: The body text of the email.    """    return f"Email delivered to {to}"

This section implements gate that tool behind human approval.

In [ ]:
# %% Step 3: Gate that tool behind human approval# interrupt_on maps tool name -> True (all four decision types allowed) or an# InterruptOnConfig dict such as {"allowed_decisions": ["approve", "reject"]}.agent = create_deep_agent(    model=MODEL,    tools=[send_email],    interrupt_on={"send_email": True},    checkpointer=InMemorySaver(),  # REQUIRED: the pause must be persisted    system_prompt="You are an assistant. Use send_email when asked. Be brief.",)

This section implements a helper that runs, inspects the pause, and resumes.

In [ ]:
# %% Step 4: A helper that runs, inspects the pause, and resumesdef run_with_decision(decision: dict, thread_id: str) -> None:    """Start a run, show what the agent wants to do, then apply `decision`.    Each call uses a FRESH thread_id — reusing one would resume the previous,    already-resolved run instead of triggering a new interrupt.    """    config = {"configurable": {"thread_id": thread_id}}    request = "Email alice@example.com saying the report is ready."    result = agent.invoke({"messages": [{"role": "user", "content": request}]}, config=config)    # The pause shows up as "__interrupt__" in the returned state.    interrupts = result.get("__interrupt__")    if not interrupts:        print("  [yellow]No interrupt fired — the agent never called the tool.[/yellow]")        return    for action in interrupts[0].value["action_requests"]:        print(f"  [bold]paused before:[/bold] [green]{action['name']}[/green]({action['args']})")    # Resume. "decisions" is a LIST, one entry per pending action request.    print(f"  [bold]human decides:[/bold] [magenta]{decision['type']}[/magenta]")    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)    print(f"  [bold]outcome:[/bold] {text_of(resumed['messages'][-1]).strip()}\n")

This section implements approve — the tool runs with the original arguments.

In [ ]:
# %% Step 5: Approve — the tool runs with the original argumentsprint("[bold cyan]1. APPROVE[/bold cyan]")run_with_decision({"type": "approve"}, thread_id="approve-run")

This section implements reject — the tool is skipped and your message goes back to the model.

In [ ]:
# %% Step 6: Reject — the tool is skipped and your message goes back to the modelprint("[bold cyan]2. REJECT[/bold cyan]")run_with_decision(    {"type": "reject", "message": "Do not email anyone. Draft it for me instead."},    thread_id="reject-run",)

This section implements the other two decision types.

In [ ]:
# %% Step 7: The other two decision types# Same mechanism, different payloads — swap either into run_with_decision():##   edit    -> run the tool, but with arguments YOU supply:#              {"type": "edit",#               "edited_action": {"name": "send_email",#                                 "args": {"to": "bob@example.com", "body": "..."}}}##   respond -> skip the tool and feed your own text back as its result:#              {"type": "respond", "message": "Already sent it myself."}print("[dim]See Step 7 in the source for the 'edit' and 'respond' payloads.[/dim]")# %%